### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# (b) & (c)

### 1: Load & Prepare Data

In [9]:
# Load weekly stock returns from two files and concatenate
stock_returns1 = pd.read_excel('D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\TRD_Week_1.xlsx', header=0, skiprows=[1, 2], dtype={'Stkcd': str})
stock_returns2 = pd.read_excel('D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\TRD_Week_2.xlsx', header=0, skiprows=[1, 2], dtype={'Stkcd': str})
stock_returns = pd.concat([stock_returns1, stock_returns2], ignore_index=True)

# Filter data before '2023-01'
stock_returns = stock_returns[stock_returns['Trdwnt'] < '2023-01'].reset_index(drop=True)

# Load risk-free rate data
rf_data = pd.read_excel('D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\weekly_risk_free_rate.xlsx', sheet_name='Sheet1')
rf_data['trading_date_yw'] = pd.to_datetime(rf_data['trading_date_yw'])
rf_data.set_index('trading_date_yw', inplace=True)
rf_data['Trdwnt'] = rf_data.index.to_period('W').strftime('%Y-%U')

<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:3: SyntaxWarning: invalid escape sequence '\M'
<>:10: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:3: SyntaxWarning: invalid escape sequence '\M'
<>:10: SyntaxWarning: invalid escape sequence '\M'
C:\Users\19927\AppData\Local\Temp\ipykernel_18220\3365625091.py:2: SyntaxWarning: invalid escape sequence '\M'
  stock_returns1 = pd.read_excel('D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\TRD_Week_1.xlsx', header=0, skiprows=[1, 2], dtype={'Stkcd': str})
C:\Users\19927\AppData\Local\Temp\ipykernel_18220\3365625091.py:3: SyntaxWarning: invalid escape sequence '\M'
  stock_returns2 = pd.read_excel('D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\TRD_Week_2.xlsx', header=0, skiprows=[1, 2], dtype={'Stkcd': str})
C:\Users\19927\AppData\Local\Temp\ipykernel_18220\3365625091.py:10: SyntaxWarning: invalid escape sequence '\M'
  rf_data = pd.read_excel('D:\MG\！CUHKSZ\~！大二 下\FIN3080\Assignments\A3\wee

### 2: Merge stock returns with risk-free rates

`Stkcd` is the stock code, `Trdwnt` is the trading week, and `Wretnd` is the weekly stock return.

In [ ]:
data = pd.merge(stock_returns, rf_data[['Trdwnt', 'risk_free_return']], on='Trdwnt', how='left')

print(data.head())

    Stkcd   Trdwnt    Wretnd  risk_free_return
0  000001  2017-01  0.003297          0.000571
1  000001  2017-02  0.003286          0.000738
2  000001  2017-03  0.006550          0.000543
3  000001  2017-04  0.011931          0.000774
4  000001  2017-05 -0.007503          0.000802


### 3: Calculate market return as the mean of weekly returns across all stocks per week

In [ ]:
market_returns = data.groupby('Trdwnt')['Wretnd'].mean().reset_index()
market_returns.rename(columns={'Wretnd': 'Market_Return'}, inplace=True)
data = pd.merge(data, market_returns, on='Trdwnt', how='left')

print(data.head())

    Stkcd   Trdwnt    Wretnd  risk_free_return  Market_Return
0  000001  2017-01  0.003297          0.000571       0.016611
1  000001  2017-02  0.003286          0.000738      -0.035048
2  000001  2017-03  0.006550          0.000543      -0.017417
3  000001  2017-04  0.011931          0.000774       0.019902
4  000001  2017-05 -0.007503          0.000802      -0.002530


### 4: Ensure only stocks present in all weeks are included

In [ ]:
stocks_per_week = data.groupby('Trdwnt')['Stkcd'].apply(set)
consistent_stocks = set.intersection(*stocks_per_week)
data = data[data['Stkcd'].isin(consistent_stocks)].reset_index(drop=True)

print(data.head())

    Stkcd   Trdwnt    Wretnd  risk_free_return  Market_Return
0  000001  2017-01  0.003297          0.000571       0.016611
1  000001  2017-02  0.003286          0.000738      -0.035048
2  000001  2017-03  0.006550          0.000543      -0.017417
3  000001  2017-04  0.011931          0.000774       0.019902
4  000001  2017-05 -0.007503          0.000802      -0.002530


# (d)

### 1: Divide Data into Three Periods

If `total_weeks` isn’t perfectly divisible by 3, the third period may have slightly more or fewer weeks, which is acceptable.

In [3]:
# Get unique trading weeks and divide into three equal periods
unique_weeks = sorted(data['Trdwnt'].unique())
total_weeks = len(unique_weeks)
period_size = total_weeks // 3

period1_weeks = unique_weeks[:period_size]
period2_weeks = unique_weeks[period_size:2*period_size]
period3_weeks = unique_weeks[2*period_size:]

# Split data into periods
period1_data = data[data['Trdwnt'].isin(period1_weeks)].copy()
period2_data = data[data['Trdwnt'].isin(period2_weeks)].copy()
period3_data = data[data['Trdwnt'].isin(period3_weeks)].copy()

### 2: Calculate Individual Stock Betas (First Period)

Use the model $r_{i,t} = \alpha_i + \beta_i r_{m,t} + \epsilon_i$, consistent with Section 4.1.1.

In [4]:
def calc_stock_beta(group):
    X = sm.add_constant(group['Market_Return'])
    y = group['Wretnd']
    model = sm.OLS(y, X, missing='drop').fit()
    return pd.Series({'beta': model.params['Market_Return']})

# Group by stock and calculate betas
stock_betas = period1_data.groupby('Stkcd').apply(calc_stock_beta).reset_index()

C:\Users\19927\AppData\Local\Temp\ipykernel_18220\2879579651.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stock_betas = period1_data.groupby('Stkcd').apply(calc_stock_beta).reset_index()


### 3: Form Ten Portfolios

Portfolios are formed by sorting $\beta_i$ into deciles。

In [5]:
# Assign stocks to deciles based on betas
stock_betas['beta_group'] = pd.qcut(stock_betas['beta'], 10, labels=False) + 1

# Merge beta groups back into period data
period2_data = pd.merge(period2_data, stock_betas[['Stkcd', 'beta_group']], on='Stkcd', how='left')
period3_data = pd.merge(period3_data, stock_betas[['Stkcd', 'beta_group']], on='Stkcd', how='left')

### 4: Calculate Portfolio Betas (Second Period) - Table 2

- Portfolio returns are equal-weighted averages.
- Regression uses $r_{p,t} - r_{f,t} = \alpha_p + \beta_p (r_{m,t} - r_{f,t}) + \epsilon_{p,t}$.
- Outputs $\alpha_p$, $\beta_p$, t-values, p-values, and $R^2$, matching Table 2’s structure.

In [6]:
# Calculate portfolio returns as equal-weighted averages
portfolio_returns_p2 = period2_data.groupby(['Trdwnt', 'beta_group'])['Wretnd'].mean().reset_index()
portfolio_data_p2 = pd.merge(portfolio_returns_p2, 
                            period2_data[['Trdwnt', 'Market_Return', 'risk_free_return']].drop_duplicates(), 
                            on='Trdwnt', how='left')

# Compute excess returns
portfolio_data_p2['portfolio_excess'] = portfolio_data_p2['Wretnd'] - portfolio_data_p2['risk_free_return']
portfolio_data_p2['market_excess'] = portfolio_data_p2['Market_Return'] - portfolio_data_p2['risk_free_return']

# Time series regression for each portfolio
table2_results = []
for group in range(1, 11):
    group_data = portfolio_data_p2[portfolio_data_p2['beta_group'] == group]
    if len(group_data) < 2:
        continue
    X = sm.add_constant(group_data['market_excess'])
    y = group_data['portfolio_excess']
    model = sm.OLS(y, X, missing='drop').fit()
    table2_results.append({
        'Portfolio': group,
        'Alpha': model.params['const'],
        'Alpha_t': model.tvalues['const'],
        'Alpha_p': model.pvalues['const'],
        'Beta': model.params['market_excess'],
        'Beta_t': model.tvalues['market_excess'],
        'Beta_p': model.pvalues['market_excess'],
        'R_squared': model.rsquared
    })

table2_df = pd.DataFrame(table2_results)
print("Table 2: Time Series Regression Results")
print(table2_df)

Table 2: Time Series Regression Results
   Portfolio     Alpha   Alpha_t   Alpha_p      Beta     Beta_t        Beta_p  \
0          1 -0.000036 -0.032946  0.973784  0.673459  20.328273  4.108630e-37   
1          2  0.000628  0.786456  0.433479  0.806966  32.950732  3.639059e-55   
2          3  0.000533  0.743194  0.459125  0.841312  38.289182  3.728752e-61   
3          4  0.000559  0.789344  0.431797  0.892307  41.089107  5.189665e-64   
4          5  0.000068  0.115784  0.908059  0.934127  52.069744  9.586499e-74   
5          6  0.000441  0.840940  0.402408  1.011989  62.973144  1.111078e-81   
6          7  0.000372  0.796354  0.427732  1.002425  69.979708  4.061909e-86   
7          8  0.000385  0.722094  0.471939  1.067248  65.331844  3.173578e-83   
8          9  0.001052  1.711731  0.090077  1.129976  59.963641  1.252960e-79   
9         10  0.000511  0.618047  0.537964  1.174070  46.365626  5.960116e-69   

   R_squared  
0   0.806731  
1   0.916438  
2   0.936744  
3   0.94

### 5: Calculate Average Excess Returns (Third Period)

Computes $\overline{r_{p,t} - r_{f,t}}$ over the third period.

In [7]:
# Calculate portfolio returns in the third period
portfolio_returns_p3 = period3_data.groupby(['Trdwnt', 'beta_group'])['Wretnd'].mean().reset_index()
portfolio_data_p3 = pd.merge(portfolio_returns_p3, 
                            period3_data[['Trdwnt', 'risk_free_return']].drop_duplicates(), 
                            on='Trdwnt', how='left')

# Compute excess returns
portfolio_data_p3['portfolio_excess'] = portfolio_data_p3['Wretnd'] - portfolio_data_p3['risk_free_return']

# Average excess returns per portfolio
avg_excess_returns = portfolio_data_p3.groupby('beta_group')['portfolio_excess'].mean().reset_index()
avg_excess_returns.rename(columns={'portfolio_excess': 'avg_excess_return'}, inplace=True)

### 6: Cross-Sectional Regression - Table 3

- Regresses $\overline{r_{p,t} - r_{f,t}} = \gamma_0 + \gamma_1 \beta_p + \epsilon_p$.
- Reports coefficients, t-statistics, p-values, $R^2$, and F-statistics, aligning with Table 3.

In [8]:
# Merge with portfolio betas from Period 2
cross_section_data = pd.merge(avg_excess_returns, table2_df[['Portfolio', 'Beta']], 
                             left_on='beta_group', right_on='Portfolio', how='left')

# Perform cross-sectional regression
X = sm.add_constant(cross_section_data['Beta'])
y = cross_section_data['avg_excess_return']
model = sm.OLS(y, X, missing='drop').fit()

# Compile results
table3_results = {
    'gamma_0': model.params['const'],
    'gamma_0_t': model.tvalues['const'],
    'gamma_0_p': model.pvalues['const'],
    'gamma_1': model.params['Beta'],
    'gamma_1_t': model.tvalues['Beta'],
    'gamma_1_p': model.pvalues['Beta'],
    'R_squared': model.rsquared,
    'F_stat': model.fvalue,
    'F_p': model.f_pvalue
}

print("Table 3: Cross-Sectional Regression Results")
for key, value in table3_results.items():
    print(f"{key}: {value}")

Table 3: Cross-Sectional Regression Results
gamma_0: 0.0003672152288612348
gamma_0_t: 0.49580296008370217
gamma_0_p: 0.6333655400069962
gamma_1: 0.0008345291577687977
gamma_1_t: 1.086875048397127
gamma_1_p: 0.30875145158746764
R_squared: 0.12866344734477186
F_stat: 1.181297370828253
F_p: 0.30875145158746875
